# Preprocessing and Feature Engineering Module
Cleans raw dataset, handles missing values, engineers features, and prepares train/test splits.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def clean_and_preprocess(df: pd.DataFrame, is_calibration_only: bool = True):
    """
    Cleans raw dataset, handles missing values, fixes anomalies,
    encodes categorical variables, and returns processed features and target.
    """
    df_clean = df.copy()

    # If filtering calibration subset for model training
    if is_calibration_only and 'CALIBRAT' in df_clean.columns:
        df_clean = df_clean[df_clean['CALIBRAT'] == 1].copy()

    # 1. Handle Negative/Anomalous Values
    if 'EQPDAYS' in df_clean.columns:
        df_clean['EQPDAYS'] = df_clean['EQPDAYS'].apply(lambda x: np.nan if pd.isna(x) or x < 0 else x)

    # 2. Impute Missing Values
    num_cols_with_na = ['REVENUE', 'MOU', 'RECCHRGE', 'DIRECTAS', 'OVERAGE', 'ROAM', 
                       'CHANGEM', 'CHANGER', 'AGE1', 'AGE2', 'EQPDAYS', 'PHONES', 'MODELS']
    for col in num_cols_with_na:
        if col in df_clean.columns:
            median_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(median_val)

    if 'CSA' in df_clean.columns:
        df_clean['CSA'] = df_clean['CSA'].fillna('UNKNOWN')

    # 3. Feature Engineering
    # Network Quality Index: combined dropped + blocked voice calls
    if 'DROPVCE' in df_clean.columns and 'BLCKVCE' in df_clean.columns:
        df_clean['TOTAL_FAILED_CALLS'] = df_clean['DROPVCE'] + df_clean['BLCKVCE']

    # Revenue Per Minute
    if 'REVENUE' in df_clean.columns and 'MOU' in df_clean.columns:
        df_clean['REV_PER_MOU'] = df_clean['REVENUE'] / (df_clean['MOU'] + 1.0)

    # 4. Separate Features and Target
    ignore_cols = ['CUSTOMER', 'CHURN', 'CHURNDEP', 'CALIBRAT']
    feature_cols = [c for c in df_clean.columns if c not in ignore_cols]

    # Frequency encode high-cardinality categorical variable 'CSA'
    if 'CSA' in df_clean.columns:
        csa_freq = df_clean['CSA'].value_counts(normalize=True).to_dict()
        df_clean['CSA_FREQ'] = df_clean['CSA'].map(csa_freq)
        feature_cols = [c for c in feature_cols if c != 'CSA']
        feature_cols.append('CSA_FREQ')

    X = df_clean[feature_cols].copy()
    y = df_clean['CHURN'].copy()

    return X, y, df_clean

def prepare_train_test_splits(X, y, test_size=0.2, random_state=42):
    """
    Splits features and target into stratified train and test sets, and scales numerical features.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

    return X_train_scaled, X_test_scaled, y_train, y_test, scaler

In [2]:
if __name__ == '__main__':
    try:
        df_raw = pd.read_csv('data/Sample Dataset.csv')
        X, y, df_clean = clean_and_preprocess(df_raw, is_calibration_only=True)
        print(f"Preprocessed Features Shape: {X.shape}")
        print(f"Target Distribution:\n{y.value_counts()}")
        X_train, X_test, y_train, y_test, scaler = prepare_train_test_splits(X, y)
        print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")
    except FileNotFoundError:
        print("Sample Dataset.csv not found. Please provide the file to run testing.")

Preprocessed Features Shape: (40000, 76)
Target Distribution:
CHURN
0    20000
1    20000
Name: count, dtype: int64


Train Shape: (32000, 76), Test Shape: (8000, 76)
